# VadCLIP V4 — Kết hợp hai nhánh có điều kiện

**Giả thuyết.** Nhánh phụ có thể sửa một số lỗi xếp hạng ở vùng điểm trung gian của nhánh V0. Dùng C-Branch làm V0 trên UCF-Crime và A-Branch (branch_2) trên XD-Violence.

**Đoạn code chính.** Hàm `conditional_fusion` trong ô **Công thức V4 dùng chung**. Với điểm V0 nằm ngoài khoảng (0,40, 0,60), V4 giữ điểm gốc. Trong khoảng này, trọng số nhánh phụ tăng tuyến tính từ 0 đến 0,10 tại điểm V0 bằng 0,50 rồi giảm tuyến tính về 0. Điểm V4 bằng điểm V0 cộng trọng số nhân với hiệu giữa điểm nhánh phụ và V0. Mức 0,50 là tâm vùng chọn trước theo thang điểm V0.

**Đối chứng.** Notebook chấm V0, phép trộn cố định 90/10 và V4 có điều kiện. Cả ba dùng điểm gốc, cùng checkpoint, prompt, feature và evaluator. Đối chứng 90/10 dùng mức trộn tối đa của V4 trên toàn bộ snippet. Báo hai chênh lệch, so V4 với V0 và so V4 với trộn 90/10. Chẩn đoán ghi số snippet được áp trọng số dương, số thực sự đổi điểm và số đoạn dương ngắn bị giảm điểm đỉnh. Cấu hình 0,40–0,60 và trọng số tối đa 0,10 được khóa cho phép thử thăm dò trên hai official test đã xem. Để chọn tham số, cần development riêng từng bộ.

## Chạy trên Kaggle

1. Import notebook này và gắn lại **VadCLIP UCF Checkpoint**, **VadCLIP UCF-Crime Test290**, **VadCLIP XD-Violence Checkpoint** và **VadCLIP XD-Violence Test 800**.
2. Bật **Internet** và chọn **GPU T4**. Chạy `RUN_MODE = "smoke"`. Khi cả hai bộ báo OK, đổi sang `"full"` và chọn **Run All**.
3. Tải `comparison.csv` và `summary.json` ở dòng **Đầu ra chung** dưới `/kaggle/working/vadclip_v4/`. Mỗi bộ có `run.json`, `predictions.npz` và log. Chỉ số chính là ROC-AUC/Ano-AUC trên UCF-Crime và frame AP trên XD-Violence.

Nguồn [VadCLIP, AAAI 2024](https://ojs.aaai.org/index.php/AAAI/article/view/28423) và [commit tác giả c41067f](https://github.com/nwpu-zxr/VadCLIP/commit/c41067f07d252efcda18008bea367886070c33b0).


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_MODE = "smoke"  # đổi sang "full" sau khi smoke đạt trên cả hai bộ

# Giữ None để tìm trong Add Input. Chỉ điền đường dẫn khi có nhiều bản cùng tên.
UCF_DATA_ROOT = None
UCF_CHECKPOINT = None
XD_DATA_ROOT = None
XD_FEATURE_ZIP = None
XD_CHECKPOINT = None
CLIP_WEIGHTS = None
V4_CONFIG = {
    "gate_points": (0.40, 0.50, 0.60),
    "max_aux_weight": 0.10,
}


## Công thức V4 dùng chung


In [ ]:
METHOD_SOURCE = r"""
def conditional_fusion(branch_scores, base_column):
    # Trả hai cột V0, trộn cố định 90/10 và trộn có điều kiện.
    scores = np.asarray(branch_scores, dtype=np.float64)
    if (scores.ndim != 2 or scores.shape[1] != 2 or not len(scores)
            or not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1))):
        raise ValueError("V4 cần hai cột điểm hữu hạn trong [0, 1].")
    if base_column not in (0, 1):
        raise ValueError("Nhánh V0 phải là cột 0 hoặc 1.")

    base = scores[:, base_column]
    auxiliary = scores[:, 1 - base_column]
    max_weight = V4_CONFIG["max_aux_weight"]
    weight = np.interp(
        base, V4_CONFIG["gate_points"], (0.0, max_weight, 0.0), left=0.0, right=0.0,
    )
    difference = auxiliary - base
    fixed = base + max_weight * difference
    conditional = base + weight * difference
    activity = {
        "total_snippets": len(base),
        "gate_active_snippets": int(np.count_nonzero(weight)),
        "changed_snippets": int(np.count_nonzero(conditional != base)),
        "mean_aux_weight": float(weight.mean()),
    }
    return np.column_stack((scores, fixed, conditional)), activity


def short_event_diagnostics(gt, base, filtered, steps_per_video, frames_per_feature):
    # Đo đổi điểm đỉnh trong đoạn dương liên tiếp dài tối đa 3 snippet quy ra frame.
    max_frames = 3 * frames_per_feature
    offsets = np.r_[0, np.cumsum(steps_per_video)] * frames_per_feature
    if (gt.shape != (offsets[-1],) or base.shape != filtered.shape
            or base.shape != (offsets[-1] // frames_per_feature,)):
        raise ValueError("Ground truth hoặc điểm không khớp offset sự kiện.")
    deltas = []
    for left, right in zip(offsets[:-1], offsets[1:], strict=True):
        changes = np.diff(np.pad(gt[left:right].astype(np.int8), 1))
        starts = np.flatnonzero(changes == 1)
        stops = np.flatnonzero(changes == -1)
        offset = left // frames_per_feature
        for start, stop in zip(starts, stops, strict=True):
            if stop - start <= max_frames:
                first = offset + start // frames_per_feature
                last = offset + (stop - 1) // frames_per_feature + 1
                deltas.append(float(filtered[first:last].max()) - float(base[first:last].max()))
    deltas = np.asarray(deltas)
    return {
        "max_frames": max_frames,
        "count": len(deltas),
        "peak_lower_count": int((deltas < 0).sum()),
        "peak_higher_count": int((deltas > 0).sum()),
        "mean_peak_delta": float(deltas.mean()) if len(deltas) else None,
    }
"""


## Hai bộ chấm đi kèm notebook

In [ ]:
UCF_RUNNER = r'''
COMMIT = "c41067f07d252efcda18008bea367886070c33b0"
UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"
EXPECTED_INPUTS = {
    "checkpoint_sha256": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",
    "split_sha256": "d621f4ccfb3e26274d215ad53b83fd1f596bb7e9f85cde69e01399db3da97c7f",
    "ground_truth_sha256": "c6448801e0f73d19adf82506cd58d60a90c4f0913e49dcc1ad1d677ee12d6301",
    "annotation_sha256": "82d581cd4685480ffda23eb0fe1bca441b672bfe99c457adca2df4cf4780c503",
    "feature_inventory_sha256": "8aa3a3c58f9bbbcc8082e024f3892e014f059374e8b52ca737ae709945210026"
}

import csv
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from collections import defaultdict
from shutil import copyfile, copyfileobj, copytree
from urllib.error import URLError
from zipfile import ZipFile

if not INPUT.is_dir():
    raise FileNotFoundError("Import notebook trên Kaggle và gắn input trước.")

def git(*args):
    return subprocess.check_output(["git", *map(str, args)], text=True).strip()


if not SRC_REPO.exists():
    SRC_REPO.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as checkout_dir:
        checkout = Path(checkout_dir) / "source"
        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)
        checkout.rename(SRC_REPO)
if git("-C", SRC_REPO, "rev-parse", "HEAD") != COMMIT:
    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} sai commit. Hãy dùng thư mục nguồn mới.")
if git("-C", SRC_REPO, "status", "--porcelain", "--untracked-files=no"):
    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} có thay đổi. Hãy dùng checkout sạch.")

packages = {
    "ftfy": "ftfy", "packaging": "packaging", "regex": "regex",
    "tqdm": "tqdm", "sklearn": "scikit-learn",
}
missing_packages = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

np = importlib.import_module("numpy")

# Modern setuptools no longer ships pkg_resources. Adapt the import in a copy.
stage_directory = tempfile.TemporaryDirectory(prefix="vadclip_ucf_v1_")
staging = Path(stage_directory.name)
runtime_src = staging / "runtime"
copytree(SRC_REPO / "src", runtime_src, dirs_exist_ok=True)
clip_py = runtime_src / "clip" / "clip.py"
clip_source = clip_py.read_text(encoding="utf-8")
old_import = "from pkg_resources import packaging"
if clip_source.count(old_import) != 1:
    raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích. Kiểm commit.")
clip_py.write_text(clip_source.replace(old_import, "import packaging.version"), encoding="utf-8")

OUTPUT.mkdir(parents=True, exist_ok=True)
print("Commit VadCLIP", COMMIT)
print("Thử nghiệm UCF-Crime V4, kết hợp hai nhánh có điều kiện")

CLASSES = (
    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",
    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",
)
FRAMES_PER_FEATURE = 16
FEATURE_DIM = 512
VIDEO_COUNT = 290
CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_split(list_root):
    """Đọc theo đúng thứ tự repo. Tính khoảng [start, end) theo frame gốc."""
    with (list_root / "ucf_CLIP_rgbtest.csv").open(newline="", encoding="utf-8") as stream:
        entries = list(csv.DictReader(stream))
    annotations = {}
    for line in (list_root / "Temporal_Anomaly_Annotation.txt").read_text().splitlines():
        if not line.strip():
            continue
        name, label, *bounds = line.split()
        video_id = Path(name).stem
        if len(bounds) != 4 or video_id in annotations:
            raise ValueError(f"Annotation không hợp lệ: {video_id}")
        intervals = []
        for start, end in zip(bounds[::2], bounds[1::2]):
            lo, hi = int(start), int(end)
            if (lo, hi) == (-1, -1):
                continue
            if not 0 <= lo < hi:
                raise ValueError(f"Khoảng frame không hợp lệ: {video_id} {(lo, hi)}")
            intervals.append((lo, hi))
        annotations[video_id] = (label, intervals)

    rows = []
    for entry in entries:
        feature_name = Path(entry["path"]).name
        match = re.fullmatch(r"(.+)__([0-9])\.npy", feature_name)
        if match is None:
            raise ValueError(f"Tên feature sai: {feature_name}")
        video_id, crop = match.groups()
        label, intervals = annotations[video_id]
        if label != entry["label"] or label not in CLASSES:
            raise ValueError(f"Nhãn không khớp giữa CSV và annotation: {video_id}")
        rows.append({
            "id": video_id, "label": label, "feature_name": feature_name,
            "crop": int(crop), "intervals": intervals,
        })
    if len(rows) != VIDEO_COUNT or len({row["id"] for row in rows}) != VIDEO_COUNT:
        raise ValueError("Cần đúng 290 ID duy nhất từ split UCF-Crime chính thức.")
    return rows



def stage_member(archive, member, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member) as source, destination.open("wb") as target:
        copyfileobj(source, target, length=1024 * 1024)
    return destination


def find_inputs(root, keys):
    """Tìm theo tên file chính thức, kể cả khi Kaggle giữ ZIP nguyên gói."""
    found = defaultdict(list)
    if root.is_file():
        archive_path = root
    elif root.is_dir():
        for folder, _, filenames in os.walk(root):
            for name in filenames:
                key = name.casefold()
                if key in keys:
                    found[key].append(Path(folder) / name)
        if found:
            return found
        archives = sorted(root.rglob("ucfclipfeatures_test290.zip"))
        if len(archives) != 1:
            raise RuntimeError(
                "Gắn Dataset feature UCF-Crime của V0 hoặc chỉ định DATA_ROOT."
            )
        archive_path = archives[0]
    else:
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {root}")

    with ZipFile(archive_path) as archive:
        for index, member in enumerate(archive.infolist()):
            name = Path(member.filename).name
            key = name.casefold()
            if not member.is_dir() and key in keys:
                destination = staging / "features" / str(index) / name
                found[key].append(stage_member(archive, member, destination))
    return found


def choose_checkpoint():
    if CHECKPOINT is not None:
        path = Path(CHECKPOINT)
        if not path.is_file():
            raise FileNotFoundError(f"Không thấy checkpoint: {path}")
    else:
        matches = sorted(INPUT.rglob("model_ucf.pth"))
        if not matches:
            matches = sorted(INPUT.rglob("vadclip_ucf_checkpoint_upload.zip"))
        if len(matches) != 1:
            raise RuntimeError("Gắn checkpoint UCF-Crime của V0 hoặc chỉ định CHECKPOINT.")
        path = matches[0]
    if path.suffix.lower() != ".zip":
        return path
    with ZipFile(path) as archive:
        members = [m for m in archive.infolist() if Path(m.filename).name == "model_ucf.pth"]
        if len(members) != 1:
            raise ValueError(f"ZIP checkpoint phải chứa đúng một model_ucf.pth: {path.name}")
        return stage_member(archive, members[0], staging / "checkpoint" / "model_ucf.pth")


def input_fingerprint(rows, list_root, checkpoint_path):
    inventory = [[row["id"], row["label"], row["steps"], row["feature_sha256"]] for row in rows]
    inventory_bytes = json.dumps(inventory, separators=(",", ":")).encode()
    return {
        "checkpoint_sha256": sha256(checkpoint_path),
        "split_sha256": sha256(list_root / "ucf_CLIP_rgbtest.csv"),
        "ground_truth_sha256": sha256(list_root / "gt_ucf.npy"),
        "annotation_sha256": sha256(list_root / "Temporal_Anomaly_Annotation.txt"),
        "feature_inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),
    }


def prepare_clip_cache(source, cache_dir):
    """Đặt file OpenAI đã xác minh vào đúng cache mà loader gốc đọc."""
    if not source.is_file():
        raise FileNotFoundError(f"Không thấy CLIP_WEIGHTS: {source}")
    if sha256(source) != CLIP_SHA256:
        raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")
    target = cache_dir / "ViT-B-16.pt"
    if target.is_file() and sha256(target) == CLIP_SHA256:
        return target
    cache_dir.mkdir(parents=True, exist_ok=True)
    temporary = cache_dir / "ViT-B-16.pt.vadclip-part"
    copyfile(source, temporary)
    if sha256(temporary) != CLIP_SHA256:
        raise ValueError("Bản sao CLIP lỗi. Kiểm dung lượng và file input.")
    temporary.replace(target)
    return target


def feature_array(path):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    if array.ndim != 2 or array.shape[1] != FEATURE_DIM or array.shape[0] < 1:
        raise ValueError(f"Feature cần dạng [T, 512], nhận {array.shape}: {path}")
    if array.dtype not in (np.float16, np.float32, np.float64):
        raise ValueError(f"Feature phải là số thực, nhận {array.dtype}: {path}")
    return array


def frame_labels(row, length):
    labels = np.zeros(length, dtype=np.uint8)
    for start, end in row["intervals"]:
        labels[start:end] = 1
    return labels


def validate_ground_truth(rows, gt):
    """Kiểm cả nội dung và thứ tự, ngoài phép kiểm tổng số frame."""
    offset = 0
    for row in rows:
        stop = offset + row["steps"] * FRAMES_PER_FEATURE
        expected = frame_labels(row, stop - offset)
        if not np.array_equal(gt[offset:stop], expected):
            raise ValueError(
                f"Ground truth lệch tại {row['id']} (frame offset {offset}, "
                f"{row['steps']} bước). Kiểm lại bản video/feature và frame count."
            )
        offset = stop
    if offset != len(gt):
        raise ValueError(f"Số frame không khớp: input={offset}, ground truth={len(gt)}")

# Kiểm đầu vào và đối chiếu hash với V0 đã chạy.
list_root = SRC_REPO / "list"
rows = read_split(list_root)
data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT
expected = {row["feature_name"].casefold(): row["feature_name"] for row in rows}
found = find_inputs(data_root, expected)
missing = sorted(expected[key] for key in expected if not found[key])
ambiguous = {
    expected[key]: [str(path) for path in sorted(found[key])]
    for key in expected if len(found[key]) > 1
}
if missing or ambiguous:
    issue_path = OUTPUT / "input_issues.json"
    issue_path.write_text(json.dumps({
        "missing": missing,
        "ambiguous": ambiguous,
    }, indent=2), encoding="utf-8")
    raise RuntimeError(
        f"Thiếu {len(missing)} file, trùng {len(ambiguous)} ID. "
        f"Xem {issue_path} và kiểm DATA_ROOT."
    )
for row in rows:
    path = found[row["feature_name"].casefold()][0]
    row["path"] = path
    row["steps"] = len(feature_array(path))
    row["feature_sha256"] = sha256(path)

gt = np.load(list_root / "gt_ucf.npy", allow_pickle=False)
if gt.ndim != 1 or set(np.unique(gt)) != {0.0, 1.0}:
    raise ValueError("Ground truth UCF cần vector nhị phân cấp frame.")
validate_ground_truth(rows, gt)
checkpoint_path = choose_checkpoint()
input_hashes = input_fingerprint(rows, list_root, checkpoint_path)
mismatches = [key for key, value in EXPECTED_INPUTS.items() if input_hashes[key] != value]
if mismatches:
    raise ValueError("Đầu vào khác lượt V0 đã đối chiếu: " + ", ".join(mismatches))
print(f"Đã khớp {len(rows)} video, {len(gt):,} frame và toàn bộ hash đầu vào với V0.")

import torch

sys.path.insert(0, str(runtime_src))
from model import CLIPVAD
from ucf_option import parser as ucf_parser
from utils.tools import get_batch_mask

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle. Mã VadCLIP gốc cần CUDA khi chạy forward.")
device = torch.device("cuda:0")
torch.cuda.set_device(device)
torch.manual_seed(234)

# Dùng đúng cấu hình UCF từ mã tác giả, không đọc cờ của kernel Jupyter.
args = ucf_parser.parse_args([])
state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
if isinstance(state, dict):
    for field in ("state_dict", "model_state_dict"):
        if field in state:
            state = state[field]
            break
if CLIP_WEIGHTS is not None:
    prepare_clip_cache(Path(CLIP_WEIGHTS), Path.home() / ".cache" / "clip")
try:
    model = CLIPVAD(
        args.classes_num, args.embed_dim, args.visual_length, args.visual_width, args.visual_head,
        args.visual_layers, args.attn_window, args.prompt_prefix, args.prompt_postfix, device,
    )
except URLError as exc:
    raise RuntimeError(
        "Không tải được CLIP ViT-B/16. Bật Internet hoặc gắn ViT-B-16.pt chính thức "
        "và đặt CLIP_WEIGHTS bằng đường dẫn file."
    ) from exc
try:
    model.load_state_dict(state, strict=True)
except (RuntimeError, TypeError) as exc:
    raise RuntimeError(
        f"Checkpoint không khớp model VadCLIP UCF-Crime: {checkpoint_path}. "
        "Kiểm lại file từ cột Model dòng UCF-Crime, khác file ViT-B-16.pt."
    ) from exc
del state
model.to(device).eval()
print("Đã nạp đầy đủ checkpoint trên", torch.cuda.get_device_name(device))

def predict_video(feature, model, device, block_length):
    """Trả điểm [T, 2], lần lượt sigmoid(C) và 1-softmax(A)[Normal]."""
    steps = len(feature)
    if steps < 1 or not np.isfinite(feature).all():
        raise ValueError("Feature rỗng hoặc chứa NaN/Inf.")
    blocks = (steps + block_length - 1) // block_length
    padded = np.zeros((blocks * block_length, FEATURE_DIM), dtype=np.float32)
    padded[:steps] = feature
    lengths = torch.tensor([
        min(block_length, steps - i * block_length) for i in range(blocks)
    ])
    visual = torch.from_numpy(padded.reshape(blocks, block_length, FEATURE_DIM)).to(device)
    mask = get_batch_mask(lengths, block_length).to(device)
    with torch.no_grad():
        _, logits_c, logits_a = model(visual, mask, list(CLASSES), lengths)
        c_score = logits_c.reshape(-1)[:steps].sigmoid()
        a_score = 1 - logits_a.reshape(-1, len(CLASSES))[:steps].softmax(dim=-1)[:, 0]
        scores = torch.stack((c_score, a_score), dim=1).cpu().numpy()
    if scores.shape != (steps, 2) or not np.isfinite(scores).all():
        raise ValueError(f"Điểm bất thường không hợp lệ: {scores.shape}")
    return scores


def run_video(row):
    feature = feature_array(row["path"])
    if feature.shape != (row["steps"], FEATURE_DIM):
        raise ValueError(f"Feature {row['id']} sai shape: {feature.shape}")
    return predict_video(feature, model, device, args.visual_length)


# Chọn theo thứ tự split và độ dài, trước khi xem điểm.
pilot = sorted({
    next(i for i, row in enumerate(rows) if row["label"] != "Normal"),
    next(i for i, row in enumerate(rows) if row["label"] == "Normal"),
    max(range(len(rows)), key=lambda i: rows[i]["steps"]),
})
torch.cuda.reset_peak_memory_stats(device)
started = time.perf_counter()
scores_by_index = {}
for i in pilot:
    before = time.perf_counter()
    scores_by_index[i] = run_video(rows[i])
    print(rows[i]["id"], scores_by_index[i].shape, f"{time.perf_counter() - before:.2f}s")
smoke_seconds = time.perf_counter() - started
print(f"Smoke test hoàn tất trong {smoke_seconds:.2f}s.")

if not SMOKE_ONLY:
    for i, row in enumerate(rows):
        if i not in scores_by_index:
            scores_by_index[i] = run_video(row)
        if (i + 1) % 25 == 0 or i + 1 == len(rows):
            print(f"Đã chạy {i + 1}/{len(rows)} video", flush=True)

from importlib.metadata import version

from sklearn.metrics import average_precision_score, roc_auc_score

SCORE_COLUMNS = ("classification", "alignment", "fixed_90_10", "conditional_v4")




def score_benchmark(rows, snippet_scores, gt):
    steps_per_video = np.asarray([row["steps"] for row in rows])
    if snippet_scores.shape != (sum(steps_per_video), len(SCORE_COLUMNS)):
        raise ValueError("Số điểm snippet hoặc cột không khớp.")
    frame_scores = np.repeat(snippet_scores, FRAMES_PER_FEATURE, axis=0)
    if len(frame_scores) != len(gt):
        raise ValueError("Số điểm cấp frame không khớp ground truth.")
    frame_mask = np.repeat(
        [row["label"] != "Normal" for row in rows],
        steps_per_video * FRAMES_PER_FEATURE,
    )
    metrics = {}
    for column, name in enumerate(SCORE_COLUMNS):
        scores = frame_scores[:, column]
        metrics[name] = {
            "roc_auc": float(roc_auc_score(gt, scores)),
            "ap": float(average_precision_score(gt, scores)),
            "ano_auc": float(roc_auc_score(gt[frame_mask], scores[frame_mask])),
        }
    return metrics


selected = sorted(scores_by_index)
if not SMOKE_ONLY and selected != list(range(len(rows))):
    raise ValueError("Thiếu dự đoán. Cần chạy toàn bộ split trước khi chấm.")
selected_rows = [rows[i] for i in selected]
ordered = [scores_by_index[i] for i in selected]
if any(score.shape != (row["steps"], 2)
       for row, score in zip(selected_rows, ordered, strict=True)):
    raise ValueError("Số điểm từng video không khớp.")
steps_per_video = [row["steps"] for row in selected_rows]
branch_scores = np.concatenate(ordered, axis=0)
scores, gate_activity = conditional_fusion(branch_scores, base_column=0)

output_dir = OUTPUT / ("smoke" if SMOKE_ONLY else "full")
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "run_kind": "smoke_inference" if SMOKE_ONLY else "full_checkpoint_inference",
    "benchmark_scope": "ucf_crime_official_test",
    "evaluation_status": "exploratory_on_previously_seen_official_test",
    "commit": COMMIT,
    "runtime_patch": "CLIP imports packaging.version directly",
    **input_hashes,
    "clip_backbone_sha256": CLIP_SHA256,
    "frames_per_feature": FRAMES_PER_FEATURE,
    "visual_block_length": args.visual_length,
    "label_order": CLASSES,
    "score_columns": SCORE_COLUMNS,
    "v4_config": {**V4_CONFIG, "baseline_column": "classification"},
    "gate_activity": gate_activity,
    "python": sys.version.split()[0],
    "libraries": {name: version(name) for name in (
        "torch", "torchvision", "numpy", "scipy", "scikit-learn", "pillow",
    )},
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(device),
    "smoke_seconds": smoke_seconds,
    "inference_seconds": time.perf_counter() - started,
    "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / (1024 ** 2),
    "video_count": len(selected_rows),
    "videos": [{key: value for key, value in row.items() if key != "path"}
               for row in selected_rows],
    "metrics": None,
}
if not SMOKE_ONLY:
    metrics = score_benchmark(rows, scores, gt)
    metadata["metrics"] = metrics
    metadata["short_events"] = short_event_diagnostics(
        gt, branch_scores[:, 0], scores[:, -1], steps_per_video, FRAMES_PER_FEATURE,
    )
    print(json.dumps(metrics, indent=2))
else:
    print("Smoke đạt, đã tính điểm V4 cho các video mẫu. Chọn full để chấm benchmark.")

np.savez_compressed(
    output_dir / "predictions.npz",
    video_id=np.asarray([row["id"] for row in selected_rows]),
    video_label=np.asarray([row["label"] for row in selected_rows]),
    snippet_offsets=np.cumsum([0] + steps_per_video),
    score_columns=np.asarray(SCORE_COLUMNS),
    scores=scores,
)
(output_dir / "run.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
stage_directory.cleanup()
print("Đầu ra", output_dir)
'''


In [ ]:
XD_RUNNER = r'''
COMMIT = "c41067f07d252efcda18008bea367886070c33b0"
UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"
VIDEO_COUNT = 800
FRAMES_PER_FEATURE = 16
FRAME_COUNT = 2_331_296
FEATURE_DIM = 512
PAPER_AP = 0.8451
PACK_MANIFEST_NAME = "xd_test_manifest.json"
PACK_ARCHIVE_NAME = "vadclip_xd_test800.zip"
PREVIOUS_PACK_ARCHIVE_NAME = "vadclip_xd_test800_kaggle_v2.zip"
PACK_MANIFEST_SHA256 = "8e5d329708857338fa94fb63a1d5733d9d8536250e6c09c3ba012f54dce997b2"
CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"

import csv
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from contextlib import ExitStack
from pathlib import PurePosixPath
from zipfile import BadZipFile, ZipFile

if RUN_MODE not in {"preflight", "smoke", "full", "v4"}:
    raise ValueError('RUN_MODE phải là "preflight", "smoke", "full" hoặc "v4".')
if not INPUT.is_dir():
    raise FileNotFoundError("Gắn dữ liệu qua Add Input trong Kaggle trước khi chạy.")

OUTPUT.mkdir(parents=True, exist_ok=True)
scratch = SRC_REPO.parent
scratch.mkdir(parents=True, exist_ok=True)
src_repo = SRC_REPO
if not src_repo.exists():
    with tempfile.TemporaryDirectory(dir=scratch) as checkout_dir:
        checkout = Path(checkout_dir) / "source"
        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)
        checkout.rename(src_repo)

def git(*args):
    return subprocess.check_output(["git", "-C", str(src_repo), *args], text=True).strip()

if git("rev-parse", "HEAD") != COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError(f"Cần checkout sạch tại commit {COMMIT}. Kiểm tra {src_repo}.")

np = importlib.import_module("numpy")

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def zip_members(path):
    try:
        with ZipFile(path) as archive:
            return [name for name in archive.namelist() if not name.endswith("/")]
    except BadZipFile as exc:
        raise ValueError(f"ZIP chưa tải đủ hoặc bị hỏng: {path.name}") from exc


def basename(value):
    return PurePosixPath(str(value).replace(chr(92), "/")).name


def read_split(path):
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not {"path", "label"} <= set(reader.fieldnames):
            raise ValueError("CSV tác giả cần cột path và label.")
        fields, rows = reader.fieldnames, list(reader)
    source_paths = [row["path"] for row in rows]
    if (
        len(rows) != VIDEO_COUNT
        or len(set(source_paths)) != VIDEO_COUNT
        or any(not path or not basename(path).endswith("__0.npy") for path in source_paths)
        or any(not row["label"] for row in rows)
    ):
        raise ValueError(f"Split XD phải gồm {VIDEO_COUNT} dòng feature hợp lệ, duy nhất.")
    return fields, rows


def matching_files(root, expected):
    found = defaultdict(list)
    if FEATURE_ZIP is None:
        if not root.is_dir():
            raise FileNotFoundError(f"DATA_ROOT không tồn tại: {root}")
        for path in root.rglob("*.npy"):
            if path.name in expected:
                found[path.name].append((path, None))
        if found:
            return found
        archives = sorted(root.rglob("*.zip"))
    else:
        archives = [Path(FEATURE_ZIP)]

    for archive_path in archives:
        if not archive_path.is_file():
            raise FileNotFoundError(f"Không thấy FEATURE_ZIP: {archive_path}")
        for member in zip_members(archive_path):
            name = basename(member)
            if name in expected:
                found[name].append((archive_path, member))
    return found


def match_one(original, candidates):
    # Nếu hai file cùng tên, chỉ phân giải khi hậu tố thư mục chính thức khớp duy nhất.
    if len(candidates) <= 1:
        return candidates
    official = PurePosixPath(original.replace(chr(92), "/")).parts
    for depth in range(2, len(official) + 1):
        matches = [
            (path, member)
            for path, member in candidates
            if PurePosixPath((member or path.as_posix()).replace(chr(92), "/")).parts[-depth:]
            == official[-depth:]
        ]
        if len(matches) == 1:
            return matches
        if not matches:
            break
    return candidates


def pick_checkpoint(root):
    if CHECKPOINT is not None:
        path = Path(CHECKPOINT)
        if not path.is_file() or path.suffix.lower() not in {".pth", ".zip"}:
            raise FileNotFoundError(f"CHECKPOINT cần đường dẫn .pth hoặc .zip hợp lệ: {path}")
        return path
    weights = [path for path in root.rglob("*.pth") if path.name in {"model_xd.pth", "model.pth"}]
    if len(weights) == 1:
        return weights[0]
    if not weights:
        archives = [
            path for path in root.rglob("*.zip")
            if any(
                basename(member) in {"model_xd.pth", "model.pth"}
                for member in zip_members(path)
            )
        ]
        if len(archives) == 1:
            return archives[0]
    raise RuntimeError(
        "Gắn checkpoint VadCLIP XD-Violence. Nếu tên khác hoặc có nhiều bản, "
        "đặt CHECKPOINT bằng đường dẫn file .pth hoặc ZIP chỉ chứa một .pth."
    )


def stage_member(archive, member, target):
    # Chỉ ghi tới tên đích do notebook tạo. Không giải nén theo đường dẫn trong ZIP.
    target.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member) as src, target.open("wb") as dest:
        shutil.copyfileobj(src, dest, length=1024 * 1024)
    return target


def checkpoint_from_input(root, staging):
    path = pick_checkpoint(root)
    if path.suffix.lower() == ".zip":
        members = [name for name in zip_members(path) if name.lower().endswith(".pth")]
        if len(members) != 1:
            raise ValueError(f"ZIP checkpoint phải chứa đúng một .pth: {path.name}")
        if basename(members[0]) == "model_ucf.pth":
            raise ValueError("ZIP đang chứa checkpoint UCF-Crime.")
        with ZipFile(path) as archive:
            path = stage_member(archive, members[0], staging / "checkpoint.pth")
    if path.name == "model_ucf.pth" or path.stat().st_size == 0:
        raise ValueError(f"File checkpoint không hợp lệ cho XD: {path.name}")
    with path.open("rb") as stream:
        if stream.read(4) == b"PK\x03\x04":
            zip_members(path)  # xác nhận có thư mục ZIP ở cuối file PyTorch
    return path


def feature_info(path):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    if (
        array.ndim != 2 or array.shape[0] == 0 or array.shape[1] != FEATURE_DIM
        or array.dtype not in (np.float16, np.float32, np.float64)
        or not np.isfinite(array).all()
    ):
        raise ValueError(f"Feature cần mảng số thực hữu hạn [T, {FEATURE_DIM}]: {path.name}")
    return len(array), sha256(path)


def read_pack_manifest(root):
    candidates = [path.read_bytes() for path in root.rglob(PACK_MANIFEST_NAME)]
    archives = set(root.rglob(PACK_ARCHIVE_NAME)) | set(root.rglob(PREVIOUS_PACK_ARCHIVE_NAME))
    if FEATURE_ZIP is not None:
        archives.add(Path(FEATURE_ZIP))
    for archive_path in sorted(archives):
        with ZipFile(archive_path) as archive:
            members = [
                name for name in archive.namelist() if basename(name) == PACK_MANIFEST_NAME
            ]
            candidates.extend(archive.read(name) for name in members)
    if not candidates:
        raise FileNotFoundError(
            "Chưa thấy manifest. Gắn Dataset VadCLIP XD-Violence Test 800 "
            f"từ {PACK_ARCHIVE_NAME} và giữ nguyên tên file ZIP."
        )
    if any(hashlib.sha256(raw).hexdigest() != PACK_MANIFEST_SHA256 for raw in candidates):
        raise ValueError("Manifest trên Kaggle không khớp bản đã khóa trong notebook.")

    manifest = json.loads(candidates[0])
    if manifest["upstream_commit"] != COMMIT:
        raise ValueError("Commit trong manifest không đúng.")
    return manifest


def check_pack_manifest(manifest, inventory, checkpoint_hash):
    if manifest["checkpoint_sha256"] != checkpoint_hash:
        raise ValueError("Sai checkpoint so với manifest đã khóa.")
    actual_features = {
        basename(row["official_path"]): {"steps": row["steps"], "sha256": row["feature_sha256"]}
        for row in inventory
    }
    if actual_features != manifest["features"]:
        names = actual_features.keys() | manifest["features"].keys()
        first_difference = next(
            name for name in sorted(names)
            if actual_features.get(name) != manifest["features"].get(name)
        )
        raise ValueError(f"Feature trên Kaggle lệch bản đã khóa: {first_difference}")
    return manifest["source_features_sha256"]


def prepare_inputs(split, gt_path, data_root, staging, upload_names=None):
    fields, rows = read_split(split)
    original_names = {basename(row["path"]) for row in rows}
    if upload_names is not None and (
        len(original_names) != VIDEO_COUNT
        or upload_names.keys() != original_names
        or len(set(upload_names.values())) != VIDEO_COUNT
    ):
        raise ValueError("Ánh xạ tên file không khớp danh sách video trong CSV gốc.")
    names = upload_names if upload_names is not None else {name: name for name in original_names}
    found = matching_files(data_root, set(names.values()))
    resolved = []
    issues = {"missing": [], "ambiguous": []}
    for row in rows:
        original = row["path"]
        candidates = match_one(original, found[names[basename(original)]])
        if len(candidates) == 0:
            issues["missing"].append(original)
        elif len(candidates) > 1:
            issues["ambiguous"].append({
                "expected": original,
                "candidates": [str(path) if member is None else f"{path.name}::{member}"
                               for path, member in candidates],
            })
        else:
            resolved.append(candidates[0])
    if any(issues.values()) or len(set(resolved)) != VIDEO_COUNT:
        report = OUTPUT / "input_issues.json"
        report.write_text(json.dumps(issues, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError(
            f"Thiếu {len(issues['missing'])} file, trùng {len(issues['ambiguous'])} dòng "
            f"hoặc dùng lại cùng feature. Xem {report}. Kiểm DATA_ROOT/FEATURE_ZIP."
        )

    gt = np.load(gt_path, allow_pickle=False)
    if gt.shape != (FRAME_COUNT,) or not np.array_equal(np.unique(gt), [0, 1]):
        raise ValueError("Ground truth XD sai shape hoặc không phải vector nhị phân.")

    inventory = []
    with ExitStack() as stack:
        archives = {}
        for index, (row, (path, member)) in enumerate(zip(rows, resolved, strict=True)):
            if member is not None:
                if path not in archives:
                    archives[path] = stack.enter_context(ZipFile(path))
                target = staging / f"{index:04d}" / basename(member)
                path = stage_member(archives[path], member, target)
            count, digest = feature_info(path)
            inventory.append({
                "official_path": row["path"], "label": row["label"],
                "steps": count, "feature_sha256": digest,
            })
            row["path"] = str(path)
    steps = sum(item["steps"] for item in inventory)
    if steps * FRAMES_PER_FEATURE != len(gt):
        raise ValueError(
            f"Feature có {steps:,} bước, cần {len(gt) // FRAMES_PER_FEATURE:,} "
            f"theo ground truth tác giả. Kiểm phiên bản gói feature và crop."
        )

    offset = 0
    for item in inventory:
        stop = offset + item["steps"] * FRAMES_PER_FEATURE
        positive_frames = int(np.count_nonzero(gt[offset:stop]))
        if "_label_A" in basename(item["official_path"]) and positive_frames:
            raise ValueError(f"Lệch ground truth trong video Normal: {item['official_path']}")
        item["positive_frames"] = positive_frames
        offset = stop

    resolved_csv = staging / "xd_CLIP_rgbtest_kaggle.csv"
    with resolved_csv.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return resolved_csv, inventory, gt

# Preflight CPU. Mỗi lần chạy tạo thư mục mới, không ghi đè input đã gắn.
stage_directory = tempfile.TemporaryDirectory(prefix="run_", dir=scratch)
staging = Path(stage_directory.name)
split = src_repo / "list" / "xd_CLIP_rgbtest.csv"
gt_path = src_repo / "list" / "gt.npy"
data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT
try:
    pack_manifest = read_pack_manifest(INPUT) if REQUIRE_PACK_MANIFEST else None
    upload_names = pack_manifest["upload_names"] if pack_manifest is not None else None
    resolved_csv, inventory, gt = prepare_inputs(split, gt_path, data_root, staging, upload_names)
    checkpoint_path = checkpoint_from_input(INPUT, staging)
    checkpoint_hash = sha256(checkpoint_path)
    source_features_hash = None
    if pack_manifest is not None:
        source_features_hash = check_pack_manifest(pack_manifest, inventory, checkpoint_hash)
except Exception:
    stage_directory.cleanup()
    raise

receipt = {
    "upstream_commit": COMMIT,
    "split_sha256": sha256(split),
    "ground_truth_sha256": sha256(gt_path),
    "checkpoint_sha256": checkpoint_hash,
    "pack_manifest_sha256": PACK_MANIFEST_SHA256 if REQUIRE_PACK_MANIFEST else None,
    "source_features_archive_sha256": source_features_hash,
    "video_count": len(inventory),
    "snippet_count": sum(row["steps"] for row in inventory),
    "frame_count": len(gt),
    "frames_per_feature": FRAMES_PER_FEATURE,
    "videos": inventory,
}
if REFERENCE_MANIFEST is not None:
    try:
        reference = json.loads(Path(REFERENCE_MANIFEST).read_text(encoding="utf-8"))
        for key in ("upstream_commit", "split_sha256", "ground_truth_sha256",
                    "checkpoint_sha256", "videos"):
            if receipt[key] != reference.get(key):
                raise ValueError(f"Bản Kaggle lệch bản tham chiếu ở trường {key}.")
    except Exception:
        stage_directory.cleanup()
        raise
receipt["matched_reference_manifest"] = REFERENCE_MANIFEST is not None

(OUTPUT / "preflight.json").write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(
    f"Preflight đạt: {len(inventory)} video, {receipt['snippet_count']:,} bước, "
    f"{len(gt):,} frame."
)
print(f"Checkpoint {checkpoint_path.name}, SHA-256 {checkpoint_hash}")
if REQUIRE_PACK_MANIFEST:
    print("SHA-256: khớp cả 800 feature và checkpoint với hai gói đã chuẩn bị.")

import contextlib
import runpy
import time
import traceback
from unittest.mock import patch




def run_upstream():
    full_run = RUN_MODE in {"full", "v4"}
    packages = {
        "ftfy": "ftfy", "packaging": "packaging", "regex": "regex", "tqdm": "tqdm",
        "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas",
    }
    missing = [
        package for module, package in packages.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

    import torch
    from sklearn import metrics

    if not torch.cuda.is_available():
        raise RuntimeError("Bật Accelerator GPU T4 trong Kaggle để chạy inference.")
    torch.cuda.set_device(0)
    torch.manual_seed(234)
    runtime = staging / "runtime"
    shutil.copytree(src_repo / "src", runtime)
    clip_source = runtime / "clip" / "clip.py"
    text = clip_source.read_text(encoding="utf-8")
    old_import = "from pkg_resources import packaging"
    if text.count(old_import) != 1:
        raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích tại commit đã khóa.")
    clip_source.write_text(text.replace(old_import, "import packaging.version"), encoding="utf-8")

    if CLIP_WEIGHTS is not None:
        source = Path(CLIP_WEIGHTS)
        if not source.is_file() or sha256(source) != CLIP_SHA256:
            raise ValueError("CLIP_WEIGHTS cần file OpenAI ViT-B-16.pt đúng SHA-256.")
        cache = Path.home() / ".cache" / "clip"
        cache.mkdir(parents=True, exist_ok=True)
        target = cache / "ViT-B-16.pt"
        if source.resolve() != target.resolve():
            with tempfile.TemporaryDirectory(dir=cache) as copy_dir:
                temporary = Path(copy_dir) / target.name
                shutil.copyfile(source, temporary)
                if sha256(temporary) != CLIP_SHA256:
                    raise ValueError("Bản sao trọng số CLIP không khớp SHA-256.")
                temporary.replace(target)

    output_root = OUTPUT / RUN_MODE
    output_root.mkdir(exist_ok=True)
    output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))
    print(f"Đang chạy evaluator gốc. Log: {output_dir / 'upstream.log'}", flush=True)
    native_ap = metrics.average_precision_score
    ap_calls = []
    upstream_completed = False
    post_ap_error = None
    checkpoint_loaded = False
    forwards = 0
    torch_load = torch.load

    class SmokeComplete(Exception):
        pass

    def load_checkpoint(file, *args, **kwargs):
        nonlocal checkpoint_loaded
        if not isinstance(file, (str, os.PathLike)) or Path(file).resolve() != checkpoint_path.resolve():
            return torch_load(file, *args, **kwargs)
        checkpoint_loaded = True
        state = torch_load(file, map_location="cpu", weights_only=True)
        for field in ("state_dict", "model_state_dict"):
            if isinstance(state, dict) and field in state:
                return state[field]
        return state

    def record_ap(y_true, y_score, *args, **kwargs):
        value = native_ap(y_true, y_score, *args, **kwargs)
        if np.asarray(y_true).shape == gt.shape and np.array_equal(y_true, gt):
            scores = np.asarray(y_score).reshape(-1)
            if scores.shape != gt.shape or not np.isfinite(scores).all():
                raise ValueError("Điểm frame từ evaluator XD sai shape hoặc chứa NaN/Inf.")
            grouped = scores.reshape(-1, FRAMES_PER_FEATURE)
            if not np.all(grouped == grouped[:, :1]):
                raise ValueError("Điểm từ evaluator không theo quy tắc lặp 16 frame.")
            ap_calls.append((float(value), grouped[:, 0].copy()))
        return value

    original_cwd = Path.cwd()
    original_path = sys.path.copy()
    original_argv = sys.argv.copy()
    original_bytecode = sys.dont_write_bytecode
    try:
        os.chdir(runtime)
        sys.path.insert(0, str(runtime))
        sys.dont_write_bytecode = True
        from model import CLIPVAD
        from xd_option import parser

        # Giữ các giá trị mặc định XD của tác giả, chỉ thay đường dẫn input.
        destinations = {action.dest for action in parser._actions}
        overrides = {
            "test_list": str(resolved_csv),
            "gt_path": str(gt_path),
            "model_path": str(checkpoint_path),
        }
        defaults = parser.parse_args([])
        for key in ("gt_segment_path", "gt_label_path"):
            default = Path(getattr(defaults, key))
            path = (src_repo / "src" / default).resolve()
            if not path.is_file():
                path = (src_repo / default).resolve()
            relative = path.relative_to(src_repo)
            git("ls-files", "--error-unmatch", relative.as_posix())
            overrides[key] = str(path)
        if not overrides.keys() <= destinations:
            raise RuntimeError(f"CLI upstream thiếu tham số: {overrides.keys() - destinations}")
        flags = []
        for action in parser._actions:
            if action.dest in overrides:
                flags.extend((action.option_strings[0], overrides[action.dest]))

        original_forward = CLIPVAD.forward

        def count_forward(self, *args, **kwargs):
            nonlocal forwards
            result = original_forward(self, *args, **kwargs)
            forwards += 1
            if RUN_MODE == "smoke" and forwards == 3:
                if any(not torch.isfinite(tensor).all() for tensor in result):
                    raise ValueError("Smoke test sinh NaN/Inf.")
                raise SmokeComplete
            return result

        sys.argv = [str(runtime / "xd_test.py"), *flags]
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        with (
            (output_dir / "upstream.log").open("w", encoding="utf-8") as log,
            contextlib.redirect_stdout(log),
            contextlib.redirect_stderr(log),
            patch.object(torch, "load", load_checkpoint),
            patch.object(metrics, "average_precision_score", record_ap),
            patch.object(CLIPVAD, "forward", count_forward),
        ):
            try:
                runpy.run_path(str(runtime / "xd_test.py"), run_name="__main__")
                upstream_completed = True
            except SmokeComplete:
                if RUN_MODE != "smoke":
                    raise
            except Exception as exc:
                if not full_run or len(ap_calls) != 2:
                    raise
                traceback.print_exc(file=log)
                post_ap_error = f"{type(exc).__name__}: {exc}"
        elapsed = time.perf_counter() - started
    finally:
        os.chdir(original_cwd)
        sys.argv = original_argv
        sys.path[:] = original_path
        sys.dont_write_bytecode = original_bytecode

    if not checkpoint_loaded or forwards == 0:
        raise RuntimeError("Chưa xác nhận upstream đã nạp checkpoint XD và chạy forward.")

    result = {
        **receipt,
        "run_kind": "smoke_inference" if RUN_MODE == "smoke" else "full_checkpoint_inference",
        "runtime_patch": "CLIP imports packaging.version directly",
        "clip_backbone_sha256": CLIP_SHA256,
        "forward_calls": forwards,
        "upstream_completed": upstream_completed,
        "post_ap_error": post_ap_error,
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "elapsed_seconds": elapsed,
        "peak_torch_vram_mib": torch.cuda.max_memory_allocated() / 2**20,
        "metrics": None,
    }
    if full_run:
        if len(ap_calls) != 2:
            raise RuntimeError(
                f"Cần hai AP cấp frame từ xd_test.py, nhận {len(ap_calls)}. "
                f"Xem {output_dir / 'upstream.log'}."
            )
        scores = np.column_stack([column for _, column in ap_calls])
        result["paper_table_1_frame_ap"] = PAPER_AP
        result["metrics"] = {
            f"branch_{index + 1}": {
                "frame_ap": ap,
                "roc_auc": float(metrics.roc_auc_score(gt, np.repeat(column, FRAMES_PER_FEATURE))),
                "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),
            }
            for index, (ap, column) in enumerate(ap_calls)
        }
        score_columns = ["branch_1", "branch_2"]
        steps_per_video = [item["steps"] for item in inventory]
        if RUN_MODE == "v4":
            if forwards != len(inventory):
                raise ValueError("Số lượt forward XD không khớp inventory test.")
            scores, result["gate_activity"] = conditional_fusion(scores, base_column=1)
            score_columns.extend(("fixed_90_10", "conditional_v4"))
            for index, name in enumerate(score_columns[2:], start=2):
                frame_scores = np.repeat(scores[:, index], FRAMES_PER_FEATURE)
                ap = float(native_ap(gt, frame_scores))
                result["metrics"][name] = {
                    "frame_ap": ap,
                    "roc_auc": float(metrics.roc_auc_score(gt, frame_scores)),
                    "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),
                }
            result["v4_config"] = {**V4_CONFIG, "baseline_column": "branch_2"}
            result["evaluation_status"] = "exploratory_on_previously_seen_official_test"
            result["short_events"] = short_event_diagnostics(
                gt, scores[:, 1], scores[:, -1], steps_per_video, FRAMES_PER_FEATURE,
            )
        result["score_columns"] = score_columns
        np.savez_compressed(
            output_dir / "predictions.npz",
            official_path=np.asarray([item["official_path"] for item in inventory]),
            video_label=np.asarray([item["label"] for item in inventory]),
            snippet_offsets=np.cumsum([0] + steps_per_video),
            score_columns=np.asarray(score_columns),
            scores=scores,
        )
    elif forwards != 3:
        raise RuntimeError("Smoke test chưa đạt ba lượt forward.")

    (output_dir / "run.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if full_run:
        if post_ap_error is not None:
            print(f"Script gốc lỗi sau khi tính đủ hai AP. Xem upstream.log: {post_ap_error}")
        for name, values in result["metrics"].items():
            print(
                f"{name}: AP {values['frame_ap'] * 100:.4f}% "
                f"(chênh {values['gap_vs_paper_percentage_points']:+.4f} điểm phần trăm)"
            )
    else:
        print("Smoke đạt ba lượt forward. Chọn full ở ô cấu hình để chấm V4.")
    print(f"Đầu ra: {output_dir}")


try:
    if RUN_MODE == "preflight":
        print("Đã kiểm input. Chọn GPU và RUN_MODE = 'smoke' để thử checkpoint.")
    else:
        run_upstream()
finally:
    stage_directory.cleanup()
'''


## Chạy tuần tự và tổng hợp kết quả

In [ ]:
import csv
import json
import math
import os
import subprocess
import sys
import tempfile


def runner_header(config):
    source = ["from pathlib import Path"]
    for key, value in config.items():
        code = f"Path({str(value)!r})" if isinstance(value, Path) else repr(value)
        source.append(f"{key} = {code}")
    return "\n".join(source)


def run_dataset(name, runner, config, work_dir):
    script = work_dir / f"{name}.py"
    script.write_text(
        runner_header(config) + "\n\n" + METHOD_SOURCE + "\n\n" + runner,
        encoding="utf-8",
    )
    console_path = config["OUTPUT"].parent / f"{name}_console.log"
    print(f"\n--- {name} ---", flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with (
        console_path.open("w", encoding="utf-8") as console,
        subprocess.Popen(
            [sys.executable, "-u", str(script)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding="utf-8", errors="replace", env=env,
        ) as process,
    ):
        for line in process.stdout:
            console.write(line)
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Xem log lỗi: {console_path}")
    return console_path


def comparison_rows(dataset, report, v4_config):
    if dataset == "UCF-Crime":
        spec = (
            ("ROC-AUC", "roc_auc", True),
            ("Ano-AUC", "ano_auc", True),
            ("AP", "ap", False),
        )
        first, second = "classification", "alignment"
        baseline = first
    else:
        spec = (("Frame AP", "frame_ap", True), ("ROC-AUC", "roc_auc", False))
        first, second = "branch_1", "branch_2"
        baseline = second
    if report["run_kind"] != "full_checkpoint_inference":
        raise ValueError("Kết quả chưa phải lượt full.")
    expected_count = 290 if dataset == "UCF-Crime" else 800
    if report["video_count"] != expected_count:
        raise ValueError("Số video trong báo cáo sai benchmark.")
    if report.get("evaluation_status") != "exploratory_on_previously_seen_official_test":
        raise ValueError("Thiếu trạng thái đánh giá V4.")
    expected_config = {
        **v4_config, "gate_points": list(v4_config["gate_points"]), "baseline_column": baseline,
    }
    if report.get("v4_config") != expected_config:
        raise ValueError("Báo cáo không khớp công thức hoặc nhánh V0 đã khóa.")
    metrics = report["metrics"]
    rows = []
    for label, key, primary in spec:
        a, b, v0, fixed, v4 = (
            metrics[name][key] for name in (first, second, baseline, "fixed_90_10", "conditional_v4")
        )
        if not all(math.isfinite(value) and 0 <= value <= 1 for value in (a, b, v0, fixed, v4)):
            raise ValueError(f"Metric ngoài miền [0, 1]: {label}")
        rows.append({
            "dataset": dataset, "metric": label, "primary": primary,
            "branch_1_pct": 100 * a, "branch_2_pct": 100 * b,
            "baseline_column": baseline, "v0_pct": 100 * v0,
            "fixed_90_10_pct": 100 * fixed, "conditional_v4_pct": 100 * v4,
            "v4_vs_v0_pp": 100 * (v4 - v0), "v4_vs_fixed_pp": 100 * (v4 - fixed),
        })
    return rows


def run_both():
    if RUN_MODE not in {"full", "smoke"}:
        raise ValueError('RUN_MODE phải là "full" hoặc "smoke".')
    if not INPUT.is_dir():
        raise FileNotFoundError("Gắn bốn Dataset bằng Add Input trước khi chạy.")
    output_root = WORK / "vadclip_v4"
    output_root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))
    shared = {
        "INPUT": INPUT, "WORK": WORK, "CLIP_WEIGHTS": CLIP_WEIGHTS,
        "V4_CONFIG": V4_CONFIG,
        "SRC_REPO": Path(tempfile.gettempdir()) / "vadclip_xd_c41067f" / "upstream",
    }
    jobs = (
        ("UCF-Crime", "ucf_crime", UCF_RUNNER, {
            **shared, "OUTPUT": output / "ucf_crime",
            "DATA_ROOT": UCF_DATA_ROOT, "CHECKPOINT": UCF_CHECKPOINT,
            "SMOKE_ONLY": RUN_MODE == "smoke",
        }),
        ("XD-Violence", "xd_violence", XD_RUNNER, {
            **shared, "OUTPUT": output / "xd_violence",
            "DATA_ROOT": XD_DATA_ROOT, "CHECKPOINT": XD_CHECKPOINT,
            "FEATURE_ZIP": XD_FEATURE_ZIP, "REFERENCE_MANIFEST": None,
            "REQUIRE_PACK_MANIFEST": True, "RUN_MODE": "v4" if RUN_MODE == "full" else "smoke",
        }),
    )
    v4_config = V4_CONFIG
    rows = []
    runs = {}
    with tempfile.TemporaryDirectory(prefix="vadclip_v4_runners_") as work_dir:
        for dataset, name, runner, config in jobs:
            log = output / f"{name}_console.log"
            try:
                run_dataset(name, runner, config, Path(work_dir))
                mode = "smoke" if RUN_MODE == "smoke" else "full"
                if dataset == "UCF-Crime":
                    candidates = [config["OUTPUT"] / mode / "run.json"]
                else:
                    xd_mode = "v4" if RUN_MODE == "full" else "smoke"
                    candidates = list((config["OUTPUT"] / xd_mode).glob("run_*/run.json"))
                if len(candidates) != 1 or not candidates[0].is_file():
                    raise RuntimeError(f"Không tìm thấy đúng một run.json cho {dataset}.")
                report_path = candidates[0]
                report = json.loads(report_path.read_text(encoding="utf-8"))
                if RUN_MODE == "full":
                    dataset_rows = comparison_rows(dataset, report, v4_config)
                elif report["run_kind"] != "smoke_inference":
                    raise ValueError("Kết quả smoke sai chế độ.")
                run = {"status": "ok", "run_json": str(report_path), "log": str(log)}
                if RUN_MODE == "full":
                    run["short_events"] = report["short_events"]
                    run["gate_activity"] = report["gate_activity"]
                    rows.extend(dataset_rows)
                runs[dataset] = run
            except (OSError, RuntimeError, ValueError, KeyError, TypeError) as exc:
                runs[dataset] = {"status": "failed", "error": str(exc), "log": str(log)}
                print(f"{dataset} lỗi: {exc}", flush=True)

    summary_path = output / "summary.json"
    summary_path.write_text(json.dumps({
        "mode": RUN_MODE, "v4_config": v4_config, "runs": runs, "metrics": rows,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    if rows:
        with (output / "comparison.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print(
            f"\n{'Benchmark':<13} {'Metric':<10} {'V0':>10} {'90/10':>10} {'V4':>10} "
            f"{'Δ V4/V0 (pp)':>13} {'Δ V4/90-10 (pp)':>16}"
        )
        for row in rows:
            print(
                f"{row['dataset']:<13} {row['metric']:<10} {row['v0_pct']:>9.4f}% "
                f"{row['fixed_90_10_pct']:>9.4f}% {row['conditional_v4_pct']:>9.4f}% "
                f"{row['v4_vs_v0_pp']:>+13.4f} {row['v4_vs_fixed_pp']:>+16.4f}"
            )
        for dataset, run in runs.items():
            if "gate_activity" in run:
                gate = run["gate_activity"]
                diag = run["short_events"]
                print(
                    f"{dataset}: V4 đổi điểm {gate['changed_snippets']}/{gate['total_snippets']} "
                    f"snippet, {diag['peak_lower_count']}/{diag['count']} đoạn dương ngắn "
                    f"(≤{diag['max_frames']} frame) giảm điểm đỉnh."
                )
    print("\nĐầu ra chung:", output)
    failed = [dataset for dataset, run in runs.items() if run["status"] != "ok"]
    if failed:
        raise RuntimeError(f"Lượt chạy lỗi ở {', '.join(failed)}. Xem {summary_path}.")


run_both()